In [1]:
# run in bash
# conda create -n scispacy_env python=3.11 -y
# conda activate scispacy_env

# pip install scispacy==0.5.5 spacy==3.7.5
# pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.5/en_core_sci_lg-0.5.5.tar.gz

# pip install ipykernel
# pip install click
# pip install pandas
# python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_lg-0.5.4.tar.gz
# python -m ipykernel install --user --name scispacy_env --display-name "Python (scispacy_env)"

In [2]:
import json
import pandas as pd
import scispacy
import spacy
from tqdm import tqdm
from scispacy.linking import EntityLinker

In [3]:
nlp = spacy.load("en_core_sci_lg")

nlp.add_pipe(
    "scispacy_linker",
    config={
        "resolve_abbreviations": True,
        "linker_name": "umls"
    }
)

linker = nlp.get_pipe("scispacy_linker")


def load_metadata_json(path, species_label):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    for gse_id, meta in data.items():
        title = meta.get("Title", "")
        summary = meta.get("Summary", "")
        text = f"{title} {summary}".strip()

        rows.append({
            "gse_id": gse_id,
            "species": species_label,
            "title": title,
            "summary": summary,
            "text": text
        })

    return pd.DataFrame(rows)


def extract_cuis(text, score_threshold=0.80):
    doc = nlp(text)
    cuis = set()

    for ent in doc.ents:
        for cui, score in ent._.kb_ents:
            if score >= score_threshold:
                cuis.add(cui)

    return cuis

/home/rfu_smith_edu/.conda/envs/scispacy_env/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
/home/rfu_smith_edu/.conda/envs/scispacy_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/rfu_smith_edu/.conda/envs/scispacy_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://

In [4]:
# 2. read data
human_df = load_metadata_json("metadata/metadata_human.json", "human")
mouse_df = load_metadata_json("metadata/metadata_mouse.json", "mouse")

# 3. generate CUIs for each GSE
# first time
# tqdm.pandas()

# human_df["cui_set"] = human_df["text"].progress_apply(extract_cuis)
# mouse_df["cui_set"] = mouse_df["text"].progress_apply(extract_cuis)

# human_df.to_pickle("metadata/human_with_cuis.pkl")
# mouse_df.to_pickle("metadata/mouse_with_cuis.pkl")

# future:
human_df = pd.read_pickle("metadata/human_with_cuis.pkl")
mouse_df = pd.read_pickle("metadata/mouse_with_cuis.pkl")

# 4. calculate CUI overlap similarity
def jaccard(a, b):
    if not a or not b:
        return 0
    return len(a & b) / len(a | b)


def containment(a, b):
    if not a or not b:
        return 0
    return len(a & b) / min(len(a), len(b))


def overlap_count(a, b):
    return len(a & b)

# 5. find highest similarity pairs and save
from pathlib import Path
import pandas as pd
from tqdm import tqdm

output_path = Path("top_human_mouse_cui_pairs.csv")

# future: if file already exists, read it
if output_path.exists():
    top_pairs = pd.read_csv(output_path)
    print(f"Loaded existing file: {output_path}")

# first time: generate and save
else:
    rows = []

    for _, h in tqdm(human_df.iterrows(), total=len(human_df)):
        h_cuis = h["cui_set"]

        for _, m in mouse_df.iterrows():
            m_cuis = m["cui_set"]
            shared = h_cuis & m_cuis

            if len(shared) == 0:
                continue

            rows.append({
                "human_gse": h["gse_id"],
                "mouse_gse": m["gse_id"],
                "jaccard": jaccard(h_cuis, m_cuis),
                "containment": containment(h_cuis, m_cuis),
                "overlap_count": len(shared),
                "shared_cuis": sorted(shared),
                "human_title": h["title"],
                "mouse_title": m["title"]
            })

    pairs_df = pd.DataFrame(rows)

    top_pairs = pairs_df.sort_values(
        ["containment", "overlap_count", "jaccard"],
        ascending=False
    )

    top_pairs.to_csv(output_path, index=False)
    print(f"Saved new file: {output_path}")

top_pairs.head(10)

Loaded existing file: top_human_mouse_cui_pairs.csv


,human_gse,mouse_gse,jaccard,containment,overlap_count,shared_cuis,human_title,mouse_title
0,GSE71511,GSE71513,1.000000,1.000000,112,"['C0003463', 'C0005967', 'C0007102', 'C0007621...",ARID1A loss impairs enhancer-mediated gene reg...,ARID1A loss impairs enhancer-mediated gene reg...
1,GSE87042,GSE87043,1.000000,1.000000,110,"['C0003463', 'C0004561', 'C0005967', 'C0007590...",The Dynamic Epigenetic Landscape of the Retina...,The Dynamic Epigenetic Landscape of the Retina...
2,GSE103658,GSE103725,1.000000,1.000000,101,"['C0001792', 'C0002976', 'C0015609', 'C0017262...",Expression changes in Melanomas pre MAPKi trea...,Expression changes in Melanomas pre MAPKi trea...
3,GSE77702,GSE77703,1.000000,1.000000,96,"['C0002736', 'C0005456', 'C0005574', 'C0008972...",Distinct and shared functions of ALS-associate...,Distinct and shared functions of ALS-associate...
4,GSE77704,GSE77703,1.000000,1.000000,96,"['C0002736', 'C0005456', 'C0005574', 'C0008972...",Distinct and shared functions of ALS-associate...,Distinct and shared functions of ALS-associate...
5,GSE81148,GSE81149,1.000000,1.000000,93,"['C0007593', 'C0007600', 'C0008546', 'C0008976...",RNASeq of MV4;11 cells transduced with scrambl...,RNASeq of MLL-AF9 cells transduced with scraml...
6,GSE102902,GSE102903,1.000000,1.000000,66,"['C0002736', 'C0003209', 'C0004112', 'C0011381...",Neuronal EphB1 induces STAT3 activation in ast...,Neuronal EphB1 induces STAT3 activation in ast...
7,GSE95516,GSE85627,1.000000,1.000000,49,"['C0008546', 'C0008922', 'C0010813', 'C0017262...",Conserved roles for murine mDUX and human DUX4...,Conserved roles for murine mDUX and human DUX4...
8,GSE105137,GSE105138,1.000000,1.000000,23,"['C0001272', 'C0205216', 'C0334094', 'C0392756...",Gene Expression Profiling of melanoma cell lin...,Gene Expression Profiling of one melanoma cell...
9,GSE81328,GSE81329,1.000000,1.000000,21,"['C0005961', 'C0005976', 'C0023467', 'C0026336...",MEIS2 is a novel oncogenic partner in AML1-ETO...,MEIS2 is a novel oncogenic partner in AML1-ETO...


In [6]:
top_pairs.head(1000).to_csv("top_1000_human_mouse_cui_pairs.csv", index=False)

## within species

In [7]:
import heapq
from tqdm import tqdm
import pandas as pd

def jaccard(a, b):
    if not a or not b:
        return 0
    return len(a & b) / len(a | b)

def containment(a, b):
    if not a or not b:
        return 0
    return len(a & b) / min(len(a), len(b))

def get_top_within_pairs(df, label, top_n=100):
    heap = []
    counter = 0
    n = len(df)

    for i in tqdm(range(n), desc=f"within {label}"):
        gse_i = df.iloc[i]
        cui_i = gse_i["cui_set"]

        for j in range(i + 1, n):
            gse_j = df.iloc[j]
            cui_j = gse_j["cui_set"]

            shared = cui_i & cui_j

            if len(shared) == 0:
                continue

            jac = jaccard(cui_i, cui_j)
            cont = containment(cui_i, cui_j)
            overlap = len(shared)

            score_key = (cont, overlap, jac)

            row = {
                "comparison": label,
                "gse_1": gse_i["gse_id"],
                "gse_2": gse_j["gse_id"],
                "containment": cont,
                "jaccard": jac,
                "overlap_count": overlap,
                "shared_cuis": sorted(shared),
                "title_1": gse_i["title"],
                "title_2": gse_j["title"],
            }

            item = (score_key, counter, row)
            counter += 1

            if len(heap) < top_n:
                heapq.heappush(heap, item)
            else:
                heapq.heappushpop(heap, item)

    rows = [item[2] for item in heap]

    return (
        pd.DataFrame(rows)
        .sort_values(["containment", "overlap_count", "jaccard"], ascending=False)
        .reset_index(drop=True)
    )

In [8]:
# human
top_human_within = get_top_within_pairs(
    human_df,
    label="within_human",
    top_n=100
)

top_human_within.to_csv("top_within_human_cui_pairs.csv", index=False)

within within_human: 100%|██████████| 3395/3395 [10:18<00:00,  5.49it/s] 


In [9]:
# mouse
top_mouse_within = get_top_within_pairs(
    mouse_df,
    label="within_mouse",
    top_n=100
)

top_mouse_within.to_csv("top_within_mouse_cui_pairs.csv", index=False)

within within_mouse: 100%|██████████| 4066/4066 [14:43<00:00,  4.60it/s] 


In [10]:
top_human_within.head(1000).to_csv("top_1000_within_human_cui_pairs.csv", index=False)
top_mouse_within.head(1000).to_csv("top_1000_within_mouse_cui_pairs.csv", index=False)

## Together

In [12]:
# all_top_pairs = pd.concat(
#     [
#         top_pairs.assign(comparison="human_mouse"),
#         top_human_within,
#         top_mouse_within
#     ],
#     ignore_index=True
# )

# all_top_pairs.to_csv("top_all_cui_similarity_pairs.csv", index=False)

top_human_mouse = top_pairs.rename(columns={
    "human_gse": "gse_1",
    "mouse_gse": "gse_2",
    "human_title": "title_1",
    "mouse_title": "title_2"
})

top_human_mouse["comparison"] = "human_mouse"

all_top_pairs = pd.concat(
    [
        top_human_mouse,
        top_human_within,
        top_mouse_within
    ],
    ignore_index=True
)

# all_top_pairs.to_csv("top_all_cui_similarity_pairs.csv", index=False)

In [14]:
all_top_pairs.head(3000).to_csv("top_3000_all_cui_similarity_pairs.csv", index=False)